# Étape 9 — Robustesse
## MASI Hybrid Forecasting System

5 axes de robustesse sur les stratégies de l'étape 8 :

- **A. Sous-périodes** (split TEST en 2 ~474j)
- **B. Cost sensitivity** (5/10/20 bps, alignement étape 6)
- **C. Seuils HMM dynamiques** (sweep T ∈ {0.3, 0.4, 0.5, 0.6, 0.7, 0.8})
- **D. Jobson-Korkie-Memmel** (différence de Sharpe pairwise, corrige DM)
- **E. Stress proxy alternatif** (variante stratégie 7 avec risk_regime au lieu de HMM=Neutral)

## 1. Exécution

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath('.'))

import pandas as pd
from IPython.display import Image, display

import importlib.util as _ilu, sys as _sys; _spec = _ilu.spec_from_file_location('e9', '../scripts/09_robustness.py'); e9 = _ilu.module_from_spec(_spec); _sys.modules['e9'] = e9; _spec.loader.exec_module(e9)
full = e9.main()

## 2. Axe A — Sharpe par sous-période

In [ ]:
sub = pd.read_csv('../outputs/etape9/subperiod_metrics.csv')
pivot = sub.pivot(index='label', columns='period', values='sharpe').round(2)
pivot = pivot[['P1 (2022-06→2024-06)', 'P2 (2024-06→2026-04)', 'FULL']]
pivot['Δ (P1−P2)'] = (pivot['P1 (2022-06→2024-06)'] - pivot['P2 (2024-06→2026-04)']).round(2)
pivot.sort_values('FULL', ascending=False)

## 3. Axe B — Cost sensitivity

In [ ]:
cs = pd.read_csv('../outputs/etape9/cost_sensitivity.csv')
pivot = cs.pivot(index='label', columns='cost_bps', values='sharpe').round(2)
pivot['Δ (5−20bps)'] = (pivot[5.0] - pivot[20.0]).round(2)
pivot.sort_values(5.0, ascending=False)

## 4. Axe C — Seuils HMM dynamiques

In [ ]:
thr = pd.read_csv('../outputs/etape9/dynamic_hmm_sweep.csv')
thr.round(3)

## 5. Axe D — Jobson-Korkie-Memmel pairwise (p-values)

In [ ]:
p_df = pd.read_csv('../outputs/etape9/jkm_pvalue_matrix.csv', index_col=0)
diff_df = pd.read_csv('../outputs/etape9/jkm_sharpe_diff_matrix.csv', index_col=0)
print('Paires SIGNIFICATIVES (p < 0.05) :')
sig = []
keys = list(p_df.index)
for i, ka in enumerate(keys):
    for j, kb in enumerate(keys):
        if i >= j: continue
        p = p_df.loc[ka, kb]
        d = diff_df.loc[ka, kb]
        if pd.notna(p) and p < 0.05:
            sig.append({'A': ka, 'B': kb, 'ΔSR_daily': round(d, 4), 'p': round(p, 4)})
pd.DataFrame(sig) if sig else 'aucune'

## 6. Axe E — Stress proxy alternatif

In [ ]:
stress = full['stress_proxy_comparison']
pd.DataFrame({
    'Strat 7 (stress=HMM Neutral)': stress['hmm_neutral_proxy'],
    'Strat 7b (stress=risk_regime=high)': stress['risk_regime_proxy'],
}).round(4)

## 7. Plots

In [ ]:
Image('../reports/figures/etape9/etape9_subperiod_sharpe.png')

In [ ]:
Image('../reports/figures/etape9/etape9_cost_sensitivity.png')

In [ ]:
Image('../reports/figures/etape9/etape9_dynamic_hmm_threshold.png')

In [ ]:
Image('../reports/figures/etape9/etape9_jkm_heatmap.png')

In [ ]:
Image('../reports/figures/etape9/etape9_robustness_scorecard.png')

## 8. Conclusion (détaillée dans `outputs/etape9/report.md`)

**HMM-gate domine tous les axes de robustesse** :
- **Temporel** : Sharpe 1.69 (P1) ≈ 1.74 (P2) — quasi identique entre les 2 sous-périodes
- **Coûts** : survives à 20 bps avec Sharpe +0.92 (seul à rester positif fort)
- **Seuil HMM** : Sharpe ∈ [1.56, 1.76] pour T ∈ [0.3, 0.8] — pas sensible au tuning
- **JKM** : seule paire significative est HMM-gate > HMM+risk-gate (ΔSR=+0.04, p=0.043)

**Findings honnêtes négatifs** :
- **VaR-budget meurt à 20 bps** (Sharpe -0.27 ; -0.23 ; -0.33) — fragile aux frais
- **CNN-LSTM nu se dégrade temporellement** : Sharpe 1.71 (P1) → 0.65 (P2) — l'instabilité justifie le gating
- **Buy & Hold s'améliore en P2** (0.54 → 1.14) car bull market — mais reste dominé full TEST

→ **Recommandation production renforcée : CNN-LSTM + HMM-gate** est robuste sur les 5 axes.